<a href="https://colab.research.google.com/github/zm-f21/MY-NLP-Week2-Text-Generation/blob/Branch8Test/retrieval_augmented_generation__rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Only run the code below when you want to clear files:
!rm -rf /content/*

In [ ]:
# @title
!pip install transformers torch accelerate

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

In [ ]:
!huggingface-cli whoami

⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
zm-f21


In [ ]:
!pip install transformers sentence-transformers -q

In [ ]:
# @title
import os
import json
import re

DATA_DIR = "/content/province_texts/"   # your folder with .txt files
OUTPUT = "/content/training_data.jsonl"

def parse_section(text, section):
    """
    Extracts a block after SECTION_NAME: until the next SECTION or end of file.
    """
    pattern = rf"{section}:\s*(.*?)(?=\n[A-Z_]+:|\Z)"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else ""

def parse_qa_pairs(text):
    """
    Finds 'Q:' and 'A:' pairs inside QA_PAIRS section.
    """
    qa_text = parse_section(text, "QA_PAIRS")
    qa_pairs = []

    qa_matches = re.findall(r"- Q:(.*?)A:(.*?)(?=- Q:|\Z)", qa_text, re.DOTALL)
    for q, a in qa_matches:
        qa_pairs.append({
            "question": q.strip(),
            "answer": a.strip()
        })

    return qa_pairs


dataset = []

for filename in os.listdir(DATA_DIR):
    if not filename.endswith(".txt"):
        continue

    with open(os.path.join(DATA_DIR, filename), "r", encoding="utf-8") as f:
        text = f.read()

    # Extract metadata
    source_title = re.search(r"SOURCE_TITLE:\s*(.*)", text)
    province = re.search(r"PROVINCE:\s*(.*)", text)
    last_updated = re.search(r"LAST_UPDATED:\s*(.*)", text)
    url = re.search(r"URL:\s*(.*)", text)

    source_title = source_title.group(1).strip() if source_title else ""
    province = province.group(1).strip() if province else ""
    last_updated = last_updated.group(1).strip() if last_updated else ""
    url = url.group(1).strip() if url else ""

    # Extract main content
    content = parse_section(text, "CONTENT")

    # Extract Q&A
    qa_pairs = parse_qa_pairs(text)

    # Convert each Q/A into a training example
    for qa in qa_pairs:
        prompt = (
            f"PROVINCE: {province}\n"
            f"LAST_UPDATED: {last_updated}\n"
            f"SOURCE: {source_title}\n"
            f"URL: {url}\n\n"
            f"QUESTION: {qa['question']}\nANSWER:"
        )

        dataset.append({
            "prompt": prompt,
            "completion": " " + qa["answer"]  # leading space recommended for GPT-2
        })

# Save JSONL for fine-tuning
with open(OUTPUT, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")

print("Dataset written to:", OUTPUT)


Dataset written to: /content/training_data.jsonl


In this notebook, you will:

1. Learn the basic principles of RAG.
2. Ask an LLM a question without RAG to see how it responds with its pre-trained knowledge.
3. Build a simple indexing system to retrieve relevant information for a query.
4. Integrate retrieval and generation to produce better, grounded responses.
5. Compare responses generated with and without RAG to understand its benefits.





# Install Dependencies
Install libraries required for the project:

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import zipfile
import os
import re
import torch

# ----------------------------- #
#  Load Lightweight Generator
# ----------------------------- #
llm = pipeline(
    'text-generation',
    model='mistralai/Mistral-7B-Instruct-v0.2',
    torch_dtype=torch.float16,
    device_map="auto",
    )

embedding_model = SentenceTransformer('nlpaueb/legal-bert-base-uncased')


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0


## Prepare a Dataset

In [ ]:
# ----------------------------- #
#  Extract ZIP of TXT files
# ----------------------------- #
zip_path = "/content/yukon.zip"
extract_folder = "/content/yukon_texts"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

In [ ]:
# ----------------------------- #
#  Helper: Parse Metadata
# ----------------------------- #

def parse_metadata_and_content(raw_text):
    """
    Splits the metadata block and content block.
    Returns (metadata_dict, content_str).
    """
    if "CONTENT:" not in raw_text:
        raise ValueError("File missing CONTENT: separator.")

    header, content = raw_text.split("CONTENT:", 1)

    metadata = {}
    lines = header.strip().split("\n")

    current_key = None
    pdf_list = []

    for line in lines:
        if ":" in line and not line.strip().startswith("-"):
            # New metadata key
            key, value = line.split(":", 1)
            key = key.strip()
            value = value.strip()
            metadata[key] = value
            current_key = key

        elif line.strip().startswith("-"):
            # PDF bullet
            pdf_list.append(line.strip())

    if pdf_list:
        metadata["PDF_LINKS"] = "\n".join(pdf_list)

    return metadata, content.strip()

In [ ]:
# ----------------------------- #
#  Load Files + Parse Metadata
# ----------------------------- #
documents = []

for root, dirs, files in os.walk(extract_folder):
    for filename in files:
        # Skip macOS metadata files (starting with '._')
        if filename.startswith('._'):
            continue

        if filename.endswith(".txt"):
            filepath = os.path.join(root, filename)

            # Read file with 'latin-1' encoding to handle potential encoding issues
            with open(filepath, "r", encoding="latin-1") as f:
                raw = f.read()

            # Parse metadata + content inside file
            metadata, content = parse_metadata_and_content(raw)

            # Split content into paragraphs
            paragraphs = [p.strip() for p in content.split("\n\n") if p.strip()]

            # Add each paragraph as its own document
            for p in paragraphs:
                documents.append({
                    "source_title": metadata.get("SOURCE_TITLE", "Unknown"),
                    "province": metadata.get("PROVINCE", "Unknown"),
                    "last_updated": metadata.get("LAST_UPDATED", "Unknown"),
                    "url": metadata.get("URL", "N/A"),
                    "pdf_links": metadata.get("PDF_LINKS", ""),
                    "text": p
                })

print("Loaded documents:", len(documents))

Loaded documents: 73


In [ ]:
# ----------------------------- #
#  Create Embeddings
# ----------------------------- #
texts = [d["text"] for d in documents]
embeddings = embedding_model.encode(texts).astype("float32")

df = pd.DataFrame(documents)
df["Embedding"] = list(embeddings)

print("Indexing complete. Total:", len(df))

Indexing complete. Total: 73


In [ ]:
# ----------------------------- #
#  Retrieval Function
# ----------------------------- #
def retrieve_with_pandas(query, top_k=2):
    query_emb = embedding_model.encode([query])[0]

    # Compute cosine similarity
    df["Similarity"] = df["Embedding"].apply(
        lambda x: np.dot(query_emb, x) / (np.linalg.norm(query_emb) * np.linalg.norm(x))
    )

    return df.sort_values("Similarity", ascending=False).head(top_k)

### Getting Responses

With everything set up, let's see how Llama responds to some sample queries.

In [ ]:
# ----------------------------- #
#  RAG Generation Function
# ----------------------------- #

def generate_with_rag(query, top_k=2):
    top_docs = retrieve_with_pandas(query, top_k)

    context = " ".join(top_docs["text"].tolist())

    input_text = f"""
Use ONLY the following context to answer the question briefly (2–3 sentences).
Do NOT guess. Do NOT add external information.

Context:
{context}

Question: {query}
"""

    response = llm(input_text, max_new_tokens=100, num_return_sequences=1)[0]['generated_text']

    # Collect metadata for user transparency
    meta = []
    for _, row in top_docs.iterrows():
        meta.append(
            f"- Province: {row['province']}\n"
            f"  Source: {row['source_title']}\n"
            f"  Updated: {row['last_updated']}\n"
            f"  URL: {row['url']}\n"
        )

    metadata_block = "\n".join(meta)

    final = f"{response.strip()}\n\nSources Used:\n{metadata_block}"
    return final

In [ ]:
# ----------------------------- #
#  Test Query
# ----------------------------- #
question = "Can my landlord increase my rent in Yukon?"
print(generate_with_rag(question))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Use ONLY the following context to answer the question briefly (2–3 sentences).
Do NOT guess. Do NOT add external information.

Context:
WHEN LANDLORDS CAN INCREASE RENT
- Rent cannot be increased for the first 12 months after a new tenancy agreement.
- After the first year, rent can only increase once a year.
- Landlords cannot use algorithmic computer technology such as Artificial Intelligence (AI) to set a rental amount.
- A Notice of rent increase must be provided at least 3 months before the increase takes effect.
- Additional rent collected for new tenants moving in is not considered a rent increase. Decisions of the Residential Tenancies Office are binding and legal documents. Once a decision or order is made, the landlord and tenant will receive a written copy. Failure to follow decisions or orders may result in lawsuits in the Yukon Supreme Court.
]

Question: Can my landlord increase my rent in Yukon?

Answer: Yes, but only after the first 12 months of the tenancy agreement, a

### Make it conversational
Let's create an interactive chat loop, where you can converse with the Llama model.

Type your questions or comments, and see how the model responds!

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["bye", "quit", "exit"]:
        print("Chatbot: Goodbye!")
        break
    generate_with_rag(user_input)

You: Do I have to pay a pet deposit in Yukon?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You: bye
Chatbot: Goodbye!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model.save_pretrained("/content/my_finetuned_model")
tokenizer.save_pretrained("/content/my_finetuned_model")


# Import Necessary Libraries

In [ ]:
question2 = "What happens if i miss a rent payment due to Canada Post work stoppage?"
print(generate_with_rag(question2))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Use ONLY the following context to answer the question briefly (2–3 sentences).
Do NOT guess. Do NOT add external information.

Context:
Paying rent during the Canada Post work stoppage
Rent must be paid in full and on time â by midnight on the day itâs due. A landlord may issue a 10 Day Notice to End Tenancy for non-payment of rent if rent is not received on time, even if a cheque has been delayed in the mail. During the Canada Post work stoppage, you should use methods of service other than mail to return a security deposit. You should return the deposit in person or arrange for the tenant to collect the deposit.

Question: What happens if i miss a rent payment due to Canada Post work stoppage?

What happens if i miss a rent payment due to Canada Post work stoppage?

A landlord will usually issue a 10 Day Notice to End Tenancy for non-payment of rent if a cheque has been delayed in the mail. During the Canada Post work stoppage, the payee will likely return the unpaid rent to the 

# Step 1: Ask LLM Without RAG

In [ ]:
# Load a pre-trained language model for text generation
llm = pipeline('text-generation', model='gpt2')  # A lightweight example for demonstration

# Step 2: Create a Simple Indexing System

In this step, we will create a **retrieval mechanism** to search for relevant information from a collection of documents. This is the foundation of the **Retriever** component in Retrieval-Augmented Generation (RAG). Here's how it works:

### **Why Do We Need an Indexing System?**
- **Efficiency**: Searching through an entire dataset every time a query is made can be computationally expensive. Indexing speeds up this process by organizing data for quick retrieval.
- **Relevance**: By creating a system that compares the query with the content of the documents, we can find the most relevant information.
- **Foundation of RAG**: The retrieved information from the index will be passed to the language model to generate grounded and specific responses.


### **What is an Index?**
An index is a data structure that allows for fast search and retrieval of information. In the context of RAG, **we use embeddings to represent the content of documents numerically**. This allows us to compute the **similarity** between a query and each document efficiently.

### **Steps to Build the Indexing System**
1. **Define a Knowledge Base (Corpus)**: This is the collection of documents or pieces of information that the system can retrieve.

2. **Generate Dense Embeddings**: Each document is converted into a dense vector representation using a **pre-trained embedding model** (e.g., SentenceTransformers). Dense embeddings capture the semantic meaning of the text, making it easier to find similar content.

3. **Create the Index**: Using pandas for indexing allows you to create a more human-readable representation of the documents and their embeddings, while still enabling retrieval.

4. **Retrieve Context**: When a query is provided, it is converted into an embedding using the same model. The index is queried to find the most relevant documents based on their similarity to the query.


## Generate Dense Embeddings and Build an Index

In [ ]:
query = "What is the rent increase limit for 2026?"
results = retrieve_with_pandas(query, top_k=5)

print("\nQuery:", query)
print("\nTop Retrieved Documents:")
print(results)


Query: What is the rent increase limit for 2026?

Top Retrieved Documents:
                                                  Text        Date  Similarity
115  The 2026 rent increase limit for residential t...  2025-08-26    0.906980
114  2025 and 2026 rent increase limit \nThe 2025 r...  2025-08-26    0.903327
293  The landlord is not allowed to do this because...  2023-12-11    0.891295
123  The landlord is not allowed to do this because...  2025-08-26    0.891295
165  Learn more about the process for move-out insp...  2025-09-26    0.869938


# Step 3: Ask Question with RAG

In [ ]:
question = "What is the rent increase limit for 2026?"
response_rag = generate_with_rag(question)
print("Response from LLM with RAG:", response_rag)

Response from LLM with RAG: Using ONLY the following context, answer the question in 2-3 concise sentences.
Do NOT add any information not present in the context.
Repeat the relevant context.
Do NOT assume or calculate numbers. Output ONLY the final answer. 

Context: 2025 and 2026 rent increase limit 
The 2025 rent increase limit for residential tenancies is 3%. The 2026 rent increase limit for residential tenancies is 2.3%.

Question: What is the rent increase limit for 2026?

Answer: The rent increase limit for 2026 for residential tenancies is 4.5%. This limit applies only to residential tenancies. The 2026 rent increase limit for residential tenancies is 2.5%

The 30-day rent cap is 1.5%. The 30-day rent cap for residential tenancies is 2.5%.

The 30-day rent cap for (Information last updated on: 2025-08-26, 2025-08-26)
